# 04 — Error Analysis (Audited)
# AI-Powered Demand Forecasting & Inventory Optimization — Milestone 11

## 1. Objective
Audit the model evaluation pipeline, establish an **apples-to-apples comparison**
between MA-28, Random Forest and XGBoost on a **common evaluation population**,
and run a full error analysis (zero-demand, segments, product/store/category
level, bias, feature importance).

**Context.** An earlier baseline run reported `MA-28 Test MAE = 1.1157` while the
latest ML experiment reported `MA-28 Test MAE = 1.1501`. Before any performance
claim, this notebook establishes exactly why those two numbers differ and which
one is valid. All metrics here are **independently recalculated** from saved
predictions — previously stored metrics are never trusted blindly.

## 2. Evaluation audit
### Why MA-28 changed from 1.1157 to 1.1501

| | Original baseline (`baseline_forecasting.py`) | Latest ML run (`ml_forecasting.py` `ma28_prediction`) |
|---|---|---|
| Formula | per date: `demand.shift(1).rolling(28).mean()` — a genuine rolling forecast | `groupby('id')['demand'].shift(1).tail(28).groupby('id').mean()` — **one constant per series** |
| Window content | 28 days strictly **before** the prediction date | last 28 days of the **entire** features dataset, ending inside the test period (April 2016) |
| Look-ahead | none | **leak**: uses future test-period demand |
| Reproduces | MAE 1.1157 | MAE 1.1501 (verified bit-for-bit against `ml_results.csv`) |

The two numbers are therefore **not comparable**. The audited, leakage-free
rolling MA-28 on the common population gives **1.1138 (test) / 1.0820 (validation)**,
which is *better* than both the 1.1157 and the 1.1501 figures. The previously
claimable ~3.3% ML improvement over MA-28 is **invalid**.

In [ ]:
import pandas as pd, numpy as np
pd.set_option('display.width', 220)

PROCESSED = '../data/processed/'
ml_results = pd.read_csv(PROCESSED + 'ml_results.csv')
print('Stored metrics from the latest ML run (MA-28 rows are the LEAKED ones):')
print(ml_results.round(4).to_string(index=False))

## 3. Common evaluation population
Only observations that all three models (MA-28, Random Forest, XGBoost) can
evaluate fairly: identical `id`, `date` and actual `demand` rows. RF/XGBoost
predictions exist only for the **test** split; validation metrics for RF/XGB are
taken from the original run's `ml_results.csv` (validation predictions were not
persisted).

In [ ]:
pop = pd.read_parquet(PROCESSED + 'evaluation_population.parquet')
PRED = {'MA-28': 'ma_28_prediction', 'Random Forest': 'random_forest_prediction',
        'XGBoost': 'xgboost_prediction'}

val = pop[pop.split == 'validation']
test = pop[pop.split == 'test']
print(f'Validation rows: {len(val):,}  ({val.date.min().date()} -> {val.date.max().date()})')
print(f'Test rows:       {len(test):,}  ({test.date.min().date()} -> {test.date.max().date()})')
print(f'Series (id):     {pop.id.nunique()}   categories: {pop.cat_id.unique().tolist()}')

# integrity checks: every test row must carry all three predictions + actual
assert test[['demand'] + list(PRED.values())].notna().all().all()
assert pop.groupby(['id', 'date']).size().max() == 1
print('\nIntegrity OK: 1 row per (id, date), all 3 predictions present on test rows.')

## 4. Model comparison (audited, recalculated)
Metrics: MAE = `mean|y-ŷ|`, RMSE = `sqrt(mean((y-ŷ)²))`,
WAPE = `Σ|y-ŷ| / Σ|y|` (same definition for every model).

In [ ]:
def metrics(df, pred):
    e = df[pred] - df['demand']
    mae = e.abs().mean(); rmse = np.sqrt((e ** 2).mean())
    wape = e.abs().sum() / df['demand'].sum()
    return mae, rmse, wape

rows = []
for split, d in [('validation', val), ('test', test)]:
    for model, col in PRED.items():
        mae, rmse, wape = metrics(d, col)
        rows.append(dict(model=model, split=split, evaluation_rows=len(d),
                         MAE=mae, RMSE=rmse, WAPE=wape))
comparison = pd.DataFrame(rows)
print(comparison.round(4).to_string(index=False))
print('\nCross-check against saved audited_model_comparison.csv:')
print(pd.read_csv(PROCESSED + 'audited_model_comparison.csv').round(4).to_string(index=False))

In [ ]:
# improvement vs MA-28 (positive % = model is better; lower is better)
base = comparison.set_index(['split', 'model'])
for model in ['Random Forest', 'XGBoost']:
    for split in ['validation', 'test']:
        b = base.loc[(split, 'MA-28')]; m = base.loc[(split, model)]
        for k in ['MAE', 'RMSE', 'WAPE']:
            imp = (b[k] - m[k]) / b[k] * 100
            tag = 'BETTER' if imp > 0 else ('worse' if imp < 0 else 'tie')
            print(f'{model:14s} {split:10s} {k:5s} improvement = {imp:+6.2f}%  ({tag})')

## 5. Zero-demand analysis (test)
Does ML systematically overpredict inactive products?

In [ ]:
t = test.copy()
zero = t[t.demand == 0]; nz = t[t.demand > 0]
print(f'Zero-demand rows: {len(zero):,} ({len(zero)/len(t):.1%})   Non-zero rows: {len(nz):,}')

def seg_table(d, label):
    out = []
    for model, col in PRED.items():
        e = d[col] - d['demand']
        out.append(dict(model=model, group=label, rows=len(d), MAE=e.abs().mean(),
                        RMSE=np.sqrt((e**2).mean()),
                        WAPE=e.abs().sum()/d['demand'].sum() if d['demand'].sum() else np.nan,
                        mean_prediction=d[col].mean(), median_prediction=d[col].median(),
                        pct_predicted_gt_0=(d[col] > 0).mean(),
                        avg_overprediction=e.mean(), median_overprediction=e.median()))
    return pd.DataFrame(out)

zero_tab = pd.concat([seg_table(zero, 'zero'), seg_table(nz, 'non_zero')])
print(zero_tab.round(4).to_string(index=False))

In [ ]:
print('Overprediction on zero-demand rows (prediction - 0):')
for model, col in PRED.items():
    op = zero[col]
    print(f'{model:14s} pct>0: {(op>0).mean():6.1%}   avg overprediction: {op.mean():.3f}   median: {op.median():.3f}')
print('\nInterpretation: ALL models, including MA-28, predict >0 on most zero-demand')
print('rows (mean prediction ~0.66 vs actual 0). ML is marginally better than MA-28')
print('here (XGB zero-demand MAE 0.664 vs MA-28 0.665); RF is marginally worse (0.681).')
print('No systematic ML-specific overprediction of inactive products beyond the')
print('baseline behaviour inherent to averaging noisy counts.')

## 6. Non-zero-demand analysis

In [ ]:
print('Non-zero-demand rows (test):')
print(seg_table(nz, 'non_zero')[['model','rows','MAE','RMSE','WAPE','mean_prediction']].round(4).to_string(index=False))
print('\nAll models UNDERFORECAST active days by ~1 unit on average (see section 13).')

## 7–9. Intermittent / Stable / Volatile demand
Uses the project's existing demand classification, rebuilt from the same
features the models consumed (zero-demand rate + variability): **Intermittent**,
**Stable**, **Volatile** per series.

In [ ]:
import sys; sys.path.insert(0, '..')

feats = pd.read_parquet(PROCESSED + 'features_dev.parquet',
                        columns=['id', 'zero_demand_rate_28', 'std_demand_28', 'mean_demand_28'])
# series-level segment rule consistent with the project's EDA classification
train_stats = feats.groupby('id').agg(zero_rate=('zero_demand_rate_28', 'mean'),
                                      cv=('std_demand_28', 'mean'))
q_cv = train_stats['cv'].quantile([1/3, 2/3])
def seg(r):
    if r['zero_rate'] > 0.5: return 'Intermittent'
    return 'Volatile' if r['cv'] > q_cv.iloc[1] else 'Stable'
train_stats['segment'] = train_stats.apply(seg, axis=1)

tt = test.merge(train_stats['segment'], on='id', how='left')
seg_rows = []
for s, d in tt.groupby('segment'):
    for model, col in PRED.items():
        e = d[col] - d['demand']
        seg_rows.append(dict(segment=s, model=model, rows=len(d), MAE=e.abs().mean(),
                             RMSE=np.sqrt((e**2).mean()),
                             WAPE=e.abs().sum()/d['demand'].sum()))
seg_tab = pd.DataFrame(seg_rows).sort_values(['segment', 'model'])
print(seg_tab.round(4).to_string(index=False))
print('\nBest model per segment (MAE):')
print(seg_tab.loc[seg_tab.groupby('segment').MAE.idxmin()][['segment','model','MAE']].round(4).to_string(index=False))

**Reading:** MA-28 is best on **Intermittent** series (tiny errors on mostly-zero
series); Random Forest is best on **Stable** and **Volatile** series, where recent
level shifts and seasonality matter. The gains on Stable/Volatile are offset by
slight losses on Intermittent, which is why the global difference is ~0.2%.

## 10. Product-level errors
Easiest (lowest MAE) and hardest (highest MAE) product/store series.

In [ ]:
prod = pd.read_csv(PROCESSED + 'audit_product_level.csv')
prod = prod[prod.split == 'test']
piv = prod.pivot_table(index='id', columns='model', values='MAE')
piv.columns = ['MAE_MA28', 'MAE_RF', 'MAE_XGB']
easy = piv.sort_values('MAE_MA28').head(10); hard = piv.sort_values('MAE_MA28').tail(10)
print('EASIEST 10 series (test MAE):'); print(easy.round(3).to_string())
print('\nHARDEST 10 series (test MAE):'); print(hard.round(3).to_string())

In [ ]:
det = pd.read_csv(PROCESSED + 'audit_product_details.csv')
hard_val = [i.rsplit('_', 1)[0] + '_validation' for i in hard.index]
hd = det[det.id.isin(hard_val)]
print('Hardest series characteristics:')
print(hd[['id','MAE_MA28','train_mean','train_std','train_max','test_mean','test_zero_rate','store_id','recent_mean_28']].round(3).to_string(index=False))
print('\nPattern: hardest series = high mean demand (5-10+/day), large std, huge historical')
print('max (up to ~93), demand level shifted UP vs train (test_mean >> train_mean).')
print('Easiest series = near-dead SKUs (99%+ zero rate), where MA-28~0 is almost optimal.')
print('ML wins consistently on the hard, high-volume series (see RF/XGB columns above).')

## 11. Store-level errors

In [ ]:
store = pd.read_csv(PROCESSED + 'audit_store_level.csv')
print(store[['store_id','model','MAE','RMSE','WAPE','mean_bias']].round(4).to_string(index=False))
print('\nCA_2 is consistently the easiest store, CA_3 the hardest for every model.')
print('Differences between models within a store are small (< 1.5% MAE).')

## 12. Category & department errors
Note: the current development subset contains only the **HOBBIES** category
(HOBBIES_1 department, 300 series) — cross-category conclusions require the full
M5 dataset.

In [ ]:
cat = pd.read_csv(PROCESSED + 'audit_category_level.csv')
dept = pd.read_csv(PROCESSED + 'audit_department_level.csv')
print(cat.round(4).to_string(index=False)); print()
print(dept.round(4).to_string(index=False))

## 13. Forecast bias (`bias = prediction - actual`)
Positive bias → overforecasting, negative → underforecasting.

In [ ]:
bias = pd.read_csv(PROCESSED + 'audit_bias.csv')
print(bias.round(4).to_string(index=False))
print('\nInterpretation:')
print('- Overall bias is tiny for all models (|mean| < 0.02) -> well calibrated on average.')
print('- Zero-demand rows: every model OVERFORECASTS (~ +0.66).')
print('- Non-zero rows: every model UNDERFORECASTS (~ -1.0): means regress toward zero.')
print('- Segments: Stable/Volatile show larger positive median bias (0.3-0.6) for all models.')

## 14. Overprediction analysis

In [ ]:
for model, col in PRED.items():
    e = test[col] - test['demand']
    over = e[e > 1]
    print(f'{model:14s} rows with overprediction > 1 unit: {len(over):7,} ({len(over)/len(test):5.1%})'
          f'   avg overprediction on those rows: {over.mean():.2f}')
print('\nOverprediction is concentrated on Intermittent series (60.7% of all test rows are')
print('zero-demand) — a structural property of count data, not a model defect.')

## 15. Feature importance

In [ ]:
fi = pd.read_csv(PROCESSED + 'feature_importance.csv')
for model in ['Random Forest', 'XGBoost']:
    top = fi[fi.model == model].nlargest(10, 'importance')
    print(f'\nTop 10 — {model}:'); print(top[['feature','importance']].round(4).to_string(index=False))
    ax = top.iloc[::-1].plot.barh(x='feature', y='importance', legend=False,
                                  title=f'{model} — top 10 features', figsize=(8, 4))

In [ ]:
focus = fi[fi.feature.isin(['mean_demand_28', 'rolling_mean_28', 'rolling_mean_14',
                            'is_weekend', 'store_id'])]
print(focus.pivot_table(index='feature', columns='model', values='importance').round(4).to_string())
print('\nWhy this makes business sense:')
print('- rolling_mean_28 / mean_demand_28 dominate: recent 28-day demand level is the single')
print('  best predictor of next-day demand — exactly what MA-28 uses, so ML largely refines')
print('  the same signal (explains the tiny overall gain).')
print('- rolling_mean_14 adds a shorter-horizon level (captures recent trend/level shifts).')
print('- is_weekend / day_of_week: HOBBIES demand peaks on weekends.')
print('- store_id: store scale differences (CA_3 > CA_1 > CA_2 volumes).')
print('Caveat: feature importance is associative, NOT causal.')

## 16. Business interpretation
- **Where ML helps:** high-volume, volatile SKUs (hardest series above) — RF cuts
  MAE ~3-6% there, and cuts RMSE on validation by 0.9%. For inventory decisions on
  A-class items this is the commercially relevant segment.
- **Where it does not:** intermittent/near-dead SKUs (60%+ of rows) — MA-28 ≈ 0 is
  already near-optimal; ML adds nothing and can even overpredict slightly.
- **Bias:** all models underforecast active days by ~1 unit → safety-stock logic
  should not assume unbiased forecasts.
- **Robustness:** on test, RF beats MA-28 on only **44%** of the 300 series and
  XGB on **49.7%** — the global averages hide that ML wins on fewer than half of
  individual series. The improvement is real but very small and uneven.

## 17. Conclusion
1. The reported `MA-28 = 1.1501` was **invalid** (look-ahead leak + frozen constant
   forecast); the apples-to-apples MA-28 is **1.1138 (test)**.
2. Audited test MAE: **RF 1.1120 < XGB 1.1136 < MA-28 1.1138** — differences of
   0.02–0.17%, far inside series-level noise.
3. On **validation** (the honest selection set) **MA-28 wins** (1.0820 vs RF 1.0860
   and XGB 1.0897).
4. **Verdict: ML does NOT genuinely beat MA-28 after the audit.** Validation
   winner = **MA-28**; the marginal test-edge of RF is not a sufficient basis to
   claim superiority. Next step (only after this audit): tune models / add lag
   features beyond the rolling level, targeting the Stable/Volatile segments.